# Specimen 01 — Parallel Workers

Goal: run several worker agents on independent subtasks at the same time instead of one after another, then merge the results. Learn `asyncio.gather` before any shared state enters the picture.

In [1]:
import os
import json
import time
import asyncio
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.AsyncAnthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'

INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

# Caps how many requests are in flight at once, regardless of how many coroutines are scheduled --
# asyncio.gather alone fires every call simultaneously, which is the fastest way to get rate-limited.
_concurrency_limit = asyncio.Semaphore(5)

async def call_model(messages, tools=None, max_tokens=1200, output_schema=None):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages, thinking={"type": "disabled"})
    if tools:
        kwargs['tools'] = tools
    if output_schema:
        kwargs['output_config'] = {"format": {"type": "json_schema", "schema": output_schema}}
    async with _concurrency_limit:
        return await client.messages.create(**kwargs)


`call_model` is now `async` and built on `AsyncAnthropic` -- `asyncio.gather` needs real awaitable coroutines to get genuine concurrency; wrapping the sync client wouldn't actually overlap requests. A `Semaphore` caps in-flight requests at 5, since `gather` alone will fire every call at once. Everything else carries forward from Phase 3-5: `thinking` disabled by default, schema-enforced JSON handoffs via `output_schema`, and a default `max_tokens` of 1200 -- Phase 5 found 800 too tight for planner/researcher-style structured output, where the model kept writing past the budget and breaking the JSON mid-generation. One more Phase 5 finding worth remembering here: `output_config`'s JSON schema does not support `maxItems` on arrays -- bound response length through the prompt (ask for an exact count, not "at most"), not the schema.

## 1. Define a worker agent over a fixed, independent task set

A small local set of 6-8 short, unrelated documents to summarize. Each summary depends on nothing but its own document -- no shared state, no ordering requirement. This independence is exactly what makes the task safe to parallelize.

In [2]:
documents = {
    "doc_ocean": "The Pacific Ocean is the largest and deepest of Earth's five oceans, covering more than 30% of the planet's surface. It contains the Mariana Trench, the deepest known point on Earth, reaching nearly 11,000 meters below sea level.",
    "doc_printing": "Johannes Gutenberg introduced the movable-type printing press to Europe around 1440. It dramatically reduced the cost of producing books and is widely credited with accelerating the spread of literacy and the Renaissance.",
    "doc_coral": "Coral reefs, despite covering less than 1% of the ocean floor, support roughly 25% of all known marine species. They are highly sensitive to temperature changes, making them an early indicator of climate stress.",
    "doc_python_lang": "Python was created by Guido van Rossum and first released in 1991. Its design philosophy emphasizes code readability, and it has become one of the most widely used languages for data science, web development, and automation.",
    "doc_roman_roads": "The Roman Empire built over 400,000 kilometers of roads, of which about 80,500 kilometers were stone-paved. These roads were critical for military movement, trade, and communication across the empire.",
    "doc_photosynthesis": "Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy stored in glucose, releasing oxygen as a byproduct. It underpins nearly all food chains on Earth.",
    "doc_apollo11": "Apollo 11 was the spaceflight that first landed humans on the Moon on July 20, 1969. Astronauts Neil Armstrong and Buzz Aldrin spent about two and a half hours outside the spacecraft while Michael Collins orbited above.",
}

WORKER_SYSTEM = "You are a summarizer agent. Summarize the given document in exactly one concise sentence."

async def summarize_document(doc_id, text):
    response = await call_model(
        messages=[{"role": "user", "content": f"{WORKER_SYSTEM}\n\nDocument:\n{text}"}],
        max_tokens=150,
    )
    summary = ''.join(b.text for b in response.content if b.type == 'text')
    return doc_id, summary, response.usage

print(f"{len(documents)} documents ready: {list(documents.keys())}")

7 documents ready: ['doc_ocean', 'doc_printing', 'doc_coral', 'doc_python_lang', 'doc_roman_roads', 'doc_photosynthesis', 'doc_apollo11']


## 2. Run all workers sequentially first, and time it

A plain `for` loop calling the (async) worker one at a time with `await`, wall-clock timed with `time.perf_counter()`. This is the baseline `asyncio.gather` gets compared against.

In [3]:
start = time.perf_counter()
sequential_results = []
for doc_id, text in documents.items():
    result = await summarize_document(doc_id, text)
    sequential_results.append(result)
sequential_elapsed = time.perf_counter() - start

for doc_id, summary, usage in sequential_results:
    print(f"[{doc_id}] {summary}")
print(f"\nSequential wall-clock time: {sequential_elapsed:.2f}s")

[doc_ocean] The Pacific Ocean, Earth's largest and deepest ocean, spans over 30% of the planet's surface and contains the Mariana Trench, the deepest known point at nearly 11,000 meters below sea level.
[doc_printing] Johannes Gutenberg's introduction of the movable-type printing press to Europe around 1440 sharply lowered book production costs and is credited with accelerating literacy and the Renaissance.
[doc_coral] Coral reefs occupy under 1% of the ocean floor yet host about 25% of known marine species, and their sensitivity to temperature shifts makes them an early warning sign of climate stress.
[doc_python_lang] Python, created by Guido van Rossum and released in 1991, emphasizes code readability and has become one of the most widely used languages for data science, web development, and automation.
[doc_roman_roads] The Roman Empire constructed more than 400,000 kilometers of roads—roughly 80,500 of them stone-paved—which were vital for military mobility, commerce, and communic

## 3. Run all workers concurrently with `asyncio.gather`, and time it

Same task set, same worker function, wrapped in `asyncio.gather(*[...])` instead of a loop. Compare the wall-clock time directly against step 2 -- this comparison is the actual point of the specimen, not just getting the code to run.

In [4]:
start = time.perf_counter()
concurrent_results = await asyncio.gather(*[summarize_document(doc_id, text) for doc_id, text in documents.items()])
concurrent_elapsed = time.perf_counter() - start

for doc_id, summary, usage in concurrent_results:
    print(f"[{doc_id}] {summary}")

print(f"\nConcurrent wall-clock time: {concurrent_elapsed:.2f}s")
print(f"Sequential was {sequential_elapsed:.2f}s -- concurrent is {sequential_elapsed / concurrent_elapsed:.1f}x faster")

[doc_ocean] The Pacific Ocean is Earth's largest and deepest ocean, spanning over 30% of the planet's surface and containing the Mariana Trench, the deepest known point at nearly 11,000 meters below sea level.
[doc_printing] Johannes Gutenberg's introduction of the movable-type printing press to Europe around 1440 sharply lowered book production costs and is credited with hastening the spread of literacy and the Renaissance.
[doc_coral] Coral reefs occupy under 1% of the ocean floor yet sustain about a quarter of known marine species, and their sensitivity to temperature shifts makes them an early warning sign of climate stress.
[doc_python_lang] Python, created by Guido van Rossum and released in 1991, emphasizes code readability and has become one of the most widely used languages for data science, web development, and automation.
[doc_roman_roads] The Roman Empire constructed more than 400,000 kilometers of roads—roughly 80,500 of them stone-paved—which were vital for military movem

## 4. Merge the results

Combine the per-document summaries into one clean combined output, preserving which summary belongs to which source document.

In [5]:
def merge_summaries(results):
    return "\n".join(f"- {doc_id}: {summary}" for doc_id, summary, _ in sorted(results, key=lambda r: r[0]))

merged = merge_summaries(concurrent_results)
print(merged)

- doc_apollo11: Apollo 11 achieved the first human Moon landing on July 20, 1969, with Neil Armstrong and Buzz Aldrin spending roughly two and a half hours on the lunar surface while Michael Collins remained in orbit.
- doc_coral: Coral reefs occupy under 1% of the ocean floor yet sustain about a quarter of known marine species, and their sensitivity to temperature shifts makes them an early warning sign of climate stress.
- doc_ocean: The Pacific Ocean is Earth's largest and deepest ocean, spanning over 30% of the planet's surface and containing the Mariana Trench, the deepest known point at nearly 11,000 meters below sea level.
- doc_photosynthesis: Photosynthesis converts light energy into chemical energy stored in glucose while releasing oxygen, forming the foundation of nearly all of Earth's food chains.
- doc_printing: Johannes Gutenberg's introduction of the movable-type printing press to Europe around 1440 sharply lowered book production costs and is credited with hastening the

## 5. Handle one worker failing without losing the rest

`asyncio.gather`'s default behavior propagates the first exception and cancels everything else. Use `return_exceptions=True`, then detect and report which task(s) failed without discarding the successful results.

In [6]:
async def summarize_document_maybe_broken(doc_id, text):
    if doc_id == "doc_roman_roads":
        raise RuntimeError(f"Simulated failure summarizing {doc_id}")
    return await summarize_document(doc_id, text)

results_with_failure = await asyncio.gather(
    *[summarize_document_maybe_broken(doc_id, text) for doc_id, text in documents.items()],
    return_exceptions=True,
)

paired = list(zip(documents.keys(), results_with_failure))
successes = [(doc_id, r) for doc_id, r in paired if not isinstance(r, Exception)]
failures = [(doc_id, r) for doc_id, r in paired if isinstance(r, Exception)]

print(f"Succeeded: {len(successes)}/{len(documents)}")
for doc_id, result in successes:
    _, summary, _ = result
    print(f"  OK   {doc_id}: {summary}")
for doc_id, exc in failures:
    print(f"  FAIL {doc_id}: {exc}")

Succeeded: 6/7
  OK   doc_ocean: The Pacific Ocean, Earth's largest and deepest ocean, spans over 30% of the planet's surface and holds the Mariana Trench, the deepest known point at nearly 11,000 meters below sea level.
  OK   doc_printing: Johannes Gutenberg's introduction of the movable-type printing press in Europe around 1440 greatly lowered book production costs and helped spread literacy and fuel the Renaissance.
  OK   doc_coral: Coral reefs occupy under 1% of the ocean floor yet sustain about a quarter of known marine species, and their sensitivity to temperature shifts makes them an early warning sign of climate stress.
  OK   doc_python_lang: Python, created by Guido van Rossum and released in 1991, emphasizes code readability and has grown into one of the most widely used languages for data science, web development, and automation.
  OK   doc_photosynthesis: Photosynthesis converts light energy into chemical energy stored as glucose while releasing oxygen, forming the found

## 6. Track cost across every concurrent call

Sum `call_cost()` over every usage object once `gather` resolves. Note this is safe precisely because asyncio is single-threaded cooperative concurrency -- there's no real parallel mutation to race against here, only interleaved awaits.

In [7]:
total_cost = sum(call_cost(usage) for _, _, usage in concurrent_results)
print(f"Total cost across {len(concurrent_results)} concurrent calls: ${total_cost:.6f}")

Total cost across 7 concurrent calls: $0.015400
